In [ ]:
# Trazabilidad F1 - Importacion de librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import holidays
from pandas.tseries.offsets import DateOffset
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
print('Librerias importadas correctamente')

In [ ]:
# Trazabilidad F2 - Carga del dataset de inferencia 2025
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

print(f'Registros cargados: {len(inferencia_df)}')
print(f'Columnas: {list(inferencia_df.columns)}')
print(f'Rango de fechas: {inferencia_df["fecha"].min()} a {inferencia_df["fecha"].max()}')
print(f'Productos: {inferencia_df["nombre"].nunique()}')
inferencia_df.head()

In [ ]:
# Trazabilidad F3 - Variables temporales numericas y binarias
inferencia_df['anio'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['es_fin_semana'] = inferencia_df['fecha'].dt.dayofweek.isin([5, 6]).astype(int)

isocalendar = inferencia_df['fecha'].dt.isocalendar()
inferencia_df['semana_del_año'] = isocalendar.week.astype(int)
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['es_inicio_mes'] = (inferencia_df['fecha'].dt.day == 1).astype(int)
inferencia_df['es_fin_mes'] = (inferencia_df['fecha'].dt.is_month_end).astype(int)
inferencia_df['es_primera_quincena'] = (inferencia_df['fecha'].dt.day <= 15).astype(int)

print('Variables temporales creadas:')
temp_cols = ['anio', 'mes', 'dia_mes', 'es_fin_semana', 'semana_del_año', 'trimestre', 'es_inicio_mes', 'es_fin_mes', 'es_primera_quincena']
print(inferencia_df[['fecha'] + temp_cols].head(10))

In [ ]:
# Trazabilidad F4 - Festivos Colombia y eventos comerciales (Black Friday, Cyber Monday)
years = sorted(inferencia_df['anio'].dropna().astype(int).unique())
co_holidays = holidays.country_holidays('CO', years=years, language='es')
holiday_lookup = {pd.Timestamp(key).normalize(): value for key, value in co_holidays.items()}
inferencia_df['es_festivo'] = inferencia_df['fecha'].dt.normalize().map(holiday_lookup).fillna('').ne('').astype(int)

def marcar_eventos_comerciales(fecha):
    thanksgiving = pd.Timestamp(fecha.year, 11, 1) + DateOffset(weekday=3, weeks=3)
    black_friday = thanksgiving + pd.Timedelta(days=1)
    cyber_monday = thanksgiving + pd.Timedelta(days=3)
    return int(fecha == black_friday), int(fecha == cyber_monday)

resultados = inferencia_df['fecha'].apply(marcar_eventos_comerciales)
inferencia_df['es_black_friday'] = resultados.apply(lambda x: x[0])
inferencia_df['es_cyber_monday'] = resultados.apply(lambda x: x[1])

print(f'Festivos encontrados: {inferencia_df["es_festivo"].sum()}')
print(f'Black Friday marcados: {inferencia_df["es_black_friday"].sum()}')
print(f'Cyber Monday marcados: {inferencia_df["es_cyber_monday"].sum()}')

In [ ]:
# Trazabilidad F5 - Variable de descuento porcentual
inferencia_df['descuento_porcentaje'] = ((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base'] * 100)
inferencia_df['descuento_porcentaje'] = inferencia_df['descuento_porcentaje'].replace([np.inf, -np.inf], np.nan).fillna(0)

print('Estadisticas de descuento porcentual:')
print(inferencia_df['descuento_porcentaje'].describe())

In [ ]:
# Trazabilidad F6 - Calculo de precio_competencia y ratio_precio
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)
inferencia_df['ratio_precio'] = np.where(
    inferencia_df['precio_competencia'] != 0,
    inferencia_df['precio_base'] / inferencia_df['precio_competencia'],
    np.nan
)

# Eliminar columnas de competidores individuales
inferencia_df = inferencia_df.drop(columns=['Amazon', 'Decathlon', 'Deporvillage'])

print(f'Despues de calcular competencia: {inferencia_df.shape}')
print(f'Columnas: {list(inferencia_df.columns)}')

In [ ]:
# Trazabilidad F7 - Lag features (1-7 dias) y media movil de 7 dias
# IMPORTANTE: Los lags se calculan ANTES de filtrar solo noviembre
# Esto asegura que el dia 1 de noviembre tenga lags correctos desde octubre

inferencia_df = inferencia_df.sort_values(['producto_id', 'fecha']).reset_index(drop=True)

for lag in range(1, 8):
    inferencia_df[f'lag_{lag}_unidades'] = inferencia_df.groupby('producto_id')['unidades_vendidas'].shift(lag)

inferencia_df['media_movil_7d_unidades'] = inferencia_df.groupby('producto_id')['unidades_vendidas'].transform(
    lambda s: s.rolling(window=7, min_periods=7).mean()
)

lag_cols = [f'lag_{lag}_unidades' for lag in range(1, 8)] + ['media_movil_7d_unidades']
print('Lags calculados (primeras filas de PROD_001):')
print(inferencia_df[inferencia_df['producto_id'] == 'PROD_001'][['fecha', 'unidades_vendidas'] + lag_cols].head(10).to_string())
print(f'\nNaN restantes en lags: {inferencia_df[lag_cols].isnull().sum().sum()}')

In [ ]:
# Trazabilidad F8 - One-Hot Encoding y alineacion de columnas con df.csv
# Crear copias de variables categoricas con sufijo _h
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

# One-hot encoding
inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'], dtype=int)

# Cargar columnas de referencia de df.csv (el nuevo df extendido)
df_ref = pd.read_csv('../data/processed/df.csv', nrows=0)
columnas_ref = list(df_ref.columns)

# Anadir columnas faltantes como 0
for col in columnas_ref:
    if col not in inferencia_df.columns:
        inferencia_df[col] = 0

# Eliminar columnas extras que no estan en df.csv
columnas_extras = [col for col in inferencia_df.columns if col not in columnas_ref]
inferencia_df = inferencia_df.drop(columns=columnas_extras)

# Reordenar columnas igual que df.csv
inferencia_df = inferencia_df[columnas_ref]

print(f'Despues de alineacion: {inferencia_df.shape}')
print(f'Columnas iguales a df.csv: {list(inferencia_df.columns) == columnas_ref}')
print(f'Total columnas: {len(inferencia_df.columns)}')

In [ ]:
# Trazabilidad F9 - Filtrado: eliminar octubre, solo noviembre + guardado CSV
registros_antes = len(inferencia_df)

# Eliminar registros de octubre
inferencia_df = inferencia_df[inferencia_df['fecha'].dt.month == 11].reset_index(drop=True)

registros_despues = len(inferencia_df)
eliminados = registros_antes - registros_despues

print(f'Registros antes del filtrado: {registros_antes}')
print(f'Registros eliminados (octubre): {eliminados}')
print(f'Registros despues del filtrado: {registros_despues}')
print(f'Shape final: {inferencia_df.shape}')
print(f'Rango de fechas: {inferencia_df["fecha"].min()} a {inferencia_df["fecha"].max()}')

# Guardar CSV
os.makedirs('../data/processed', exist_ok=True)
inferencia_df.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)
print(f'\nGuardado en data/processed/inferencia_df_transformado.csv')
print(f'\nColumnas del dataset final:')
for i, c in enumerate(inferencia_df.columns):
    print(f'  {i}: {c}')

In [ ]:
# Trazabilidad F10 - Verificacion: cargar modelo y verificar compatibilidad de features
# Cargar modelo final
model_path = '../models/modelo_final.joblib'
if not os.path.exists(model_path):
    model_path = 'models/modelo_final.joblib'

modelo = joblib.load(model_path)
print(f'Modelo cargado: {type(modelo).__name__}')
print(f'Features esperadas por el modelo: {len(modelo.feature_names_in_)}')

# Verificar que inferencia_df tiene las features correctas
feature_cols = list(modelo.feature_names_in_)
missing = [f for f in feature_cols if f not in inferencia_df.columns]
extra = [c for c in inferencia_df.columns if c in feature_cols]

print(f'\nFeatures en inferencia_df: {len(extra)}')
print(f'Features faltantes: {missing}')
print(f'Todas las features presentes: {len(missing) == 0}')

# Preparar X para prediccion
X_inf = inferencia_df[feature_cols].copy()
print(f'\nShape de X_inf: {X_inf.shape}')
print(f'Primeras features: {feature_cols[:10]}')

In [ ]:
# Trazabilidad F11 - Prediccion basica del modelo para todos los productos
predicciones_basicas = modelo.predict(X_inf)
inferencia_df['unidades_predichas'] = predicciones_basicas

print('Predicciones realizadas:')
print(f'  Total registros: {len(predicciones_basicas)}')
print(f'  Media: {predicciones_basicas.mean():.2f}')
print(f'  Min: {predicciones_basicas.min():.2f}')
print(f'  Max: {predicciones_basicas.max():.2f}')

# Resumen por producto
resumen = inferencia_df.groupby('nombre').agg({
    'unidades_predichas': 'sum',
    'precio_venta': 'mean',
    'precio_competencia': 'mean'
}).round(2).sort_values('unidades_predichas', ascending=False)
print('\nResumen de predicciones por producto:')
print(resumen)